# 01. Data Loading & Preprocessing

## Agentic AI-Based Dynamic Tariff Optimization for EV Charging Networks
### Open Project 2026 — Society of Business

---

## Notebook Overview

This notebook covers the complete data ingestion and preprocessing pipeline for two datasets:

- **ACN-Data** (Adaptive Charging Network) — 16,304 EV charging sessions from JPL site, Caltech
- **UrbanEV Dataset (ST-EVCDP)** — Large-scale urban charging data from Shenzhen, China

### Steps Covered
1. Library imports and configuration
2. ACN data loading and conversion
3. UrbanEV multi-file loading and integration
4. Feature engineering (utilization rate, revenue, queue proxy, occupancy density)
5. Data cleaning with documented assumptions
6. Saving processed outputs for downstream notebooks

In [1]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries loaded successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries loaded successfully.
Pandas version: 2.3.3
NumPy version: 1.26.4


---
## 1. ACN Dataset

**Source:** Adaptive Charging Network — JPL Site, Caltech  
**Format:** JSON (delivered as `.json.xlsx`); converted to CSV for analysis  
**Coverage:** EV charging sessions with timestamps, energy delivered, session duration, and station IDs  

**Key decision:** Columns `_meta`, `end`, `start`, `_items` were entirely NaN — artifacts of JSON-to-Excel conversion. Dropped from analysis.

In [2]:
# ============================================================
# 2. ACN DATA — LOADING AND CONVERSION
# ============================================================

acn_raw = pd.read_excel("../data/acndata_sessions.json.xlsx")
print(f"Raw ACN shape: {acn_raw.shape}")
print(f"Columns: {acn_raw.columns.tolist()}")
print(acn_raw.head(3))
print("\nNull counts:\n", acn_raw.isnull().sum())

Raw ACN shape: (16304, 27)
Columns: ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']
   _meta  end  min_kWh     site  start  _items                       _id  \
0    NaN  NaN      NaN  caltech    NaN     NaN  5bc90cb9f9af8b0d7fe77cd2   
1    NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd3   
2    NaN  NaN      NaN      NaN    NaN     NaN  5bc90cb9f9af8b0d7fe77cd4   

   clusterID                 connectionTime                 disconnectTime  \
0    39.0000  Wed, 25 Apr 2018 11:08:04 GMT  Wed, 25 Apr 2018 13:20:10 GMT   
1    39.0000  Wed, 25 Apr 2018 13:45:10 GMT  Thu, 26 Apr 2018 00:56:16 GMT   
2    39.0000  Wed, 25 Apr 2018 13:45:50 GMT  Wed, 25 Apr 2018

In [3]:
# Retain only analytically meaningful columns
# Dropped: _meta, end, start, _items (100% NaN — JSON conversion artifacts)
# Dropped: kWhRequested (78% NaN — too sparse to use)

acn = acn_raw[[
    "_id", "clusterID", "stationID", "siteID",
    "connectionTime", "disconnectTime", "doneChargingTime",
    "kWhDelivered"
]].copy()

# Parse timestamps
for col in ["connectionTime", "disconnectTime", "doneChargingTime"]:
    acn[col] = pd.to_datetime(acn[col], utc=True, errors="coerce")

# Session duration in hours
acn["session_hours"] = (
    acn["disconnectTime"] - acn["connectionTime"]
).dt.total_seconds() / 3600

# Actual charging time (connection → done charging)
acn["charge_hours"] = (
    acn["doneChargingTime"] - acn["connectionTime"]
).dt.total_seconds() / 3600

# Idle time = plugged in but not actively charging
acn["idle_hours"] = acn["session_hours"] - acn["charge_hours"]

# Time features
acn["hour"] = acn["connectionTime"].dt.hour
acn["day_of_week"] = acn["connectionTime"].dt.dayofweek  # 0 = Monday
acn["is_weekend"] = acn["day_of_week"].isin([5, 6]).astype(int)

print(f"ACN shape after column selection: {acn.shape}")
print(acn.dtypes)

ACN shape after column selection: (16304, 14)
_id                              object
clusterID                       float64
stationID                        object
siteID                          float64
connectionTime      datetime64[ns, UTC]
disconnectTime      datetime64[ns, UTC]
doneChargingTime    datetime64[ns, UTC]
kWhDelivered                    float64
session_hours                   float64
charge_hours                    float64
idle_hours                      float64
hour                            float64
day_of_week                     float64
is_weekend                        int64
dtype: object


---
### 1.1 ACN Data Cleaning

**Documented assumptions and decisions:**

| Decision | Reason |
|---|---|
| Drop rows where `connectionTime` is null | 1,305 rows — no usable time reference |
| Drop sessions > 24 hours | Likely data entry errors; not operationally meaningful |
| Drop sessions with 0 or negative duration | Physically impossible |
| Drop rows where `kWhDelivered` is null | Cannot compute revenue or utilization without energy data |
| Clip negative `charge_hours` to 0 | Small negative values caused by `doneChargingTime` slightly after `disconnectTime` — data entry rounding |
| Clip negative `idle_hours` to 0 | Same rounding artifact |

In [4]:
# ============================================================
# 3. ACN DATA CLEANING
# ============================================================

before = len(acn)

# Drop rows with no connection time
acn = acn.dropna(subset=["connectionTime"])
print(f"Dropped {before - len(acn)} rows with null connectionTime")

# Drop sessions over 24 hours
acn = acn[acn["session_hours"] <= 24]
print(f"Remaining after removing sessions > 24hrs: {len(acn)}")

# Drop zero or negative duration sessions
acn = acn[acn["session_hours"] > 0]

# Drop rows with no energy delivered
acn = acn.dropna(subset=["kWhDelivered"])

# Clip negative values (rounding artifacts)
acn["charge_hours"] = acn["charge_hours"].clip(lower=0)
acn["idle_hours"] = acn["idle_hours"].clip(lower=0)

print(f"\nFinal ACN shape: {acn.shape}")
print(f"\nNull counts:\n{acn.isnull().sum()}")
print(f"\nSession hours summary:\n{acn['session_hours'].describe().round(3)}")

Dropped 1305 rows with null connectionTime
Remaining after removing sessions > 24hrs: 14848

Final ACN shape: (14848, 14)

Null counts:
_id                 0
clusterID           0
stationID           0
siteID              0
connectionTime      0
disconnectTime      0
doneChargingTime    8
kWhDelivered        0
session_hours       0
charge_hours        8
idle_hours          8
hour                0
day_of_week         0
is_weekend          0
dtype: int64

Session hours summary:
count   14848.0000
mean        5.4950
std         3.9650
min         0.0880
25%         2.0080
50%         4.6960
75%         8.6750
max        23.9910
Name: session_hours, dtype: float64


---
### 1.2 ACN Feature Engineering

The following economically meaningful features are derived from raw session data:

- **Revenue per Session** — kWh delivered × ₹15/kWh fixed baseline tariff
- **Energy Cost per kWh** — assumed ₹6/kWh (standard India grid procurement cost)
- **Profit per Session** — revenue minus energy cost
- **Queue Length Proxy** — rolling session count per cluster (approximates concurrent demand)
- **Weekend flag** — used to distinguish fleet (weekday) vs personal (weekend) usage

In [5]:
# ============================================================
# 4. ACN FEATURE ENGINEERING
# ============================================================

# Revenue and cost features
acn["revenue_per_session"] = acn["kWhDelivered"] * 15        # ₹15/kWh baseline
acn["energy_cost_per_kwh"] = 6.0                             # ₹6/kWh grid cost assumption
acn["profit_per_session"]  = acn["kWhDelivered"] * (15 - 6) # ₹9/kWh margin

# Queue length proxy — rolling session count per cluster
acn_sorted = acn.sort_values("connectionTime")
acn["queue_length_proxy"] = acn.groupby("clusterID")["connectionTime"].transform(
    lambda x: x.expanding().count()
) % 10

# Site type label
acn["site_type"] = acn["siteID"].map({1.0: "Caltech", 2.0: "JPL"}).fillna("Other")

# Weekend label
acn["usage_type"] = acn["is_weekend"].map({0: "Weekday (Fleet)", 1: "Weekend (Personal)"})

print("ACN engineered features summary:")
print(acn[[
    "revenue_per_session", "energy_cost_per_kwh",
    "profit_per_session", "queue_length_proxy"
]].describe().round(2))

# Save ACN as CSV (converted from JSON/XLSX format)
acn.to_csv("../data/acn_processed.csv", index=False)
print(f"\nACN saved. Final shape: {acn.shape}")

ACN engineered features summary:
       revenue_per_session  energy_cost_per_kwh  profit_per_session  \
count           14848.0000           14848.0000          14848.0000   
mean              134.1400               6.0000             80.4900   
std               104.3100               0.0000             62.5900   
min                 7.5200               6.0000              4.5100   
25%                59.7900               6.0000             35.8700   
50%               111.2600               6.0000             66.7600   
75%               197.9700               6.0000            118.7800   
max              1040.6000               6.0000            624.3600   

       queue_length_proxy  
count          14848.0000  
mean               4.5000  
std                2.8700  
min                0.0000  
25%                2.0000  
50%                4.5000  
75%                7.0000  
max                9.0000  

ACN saved. Final shape: (14848, 20)


---
## 2. UrbanEV Dataset (ST-EVCDP)

**Source:** Intelligent Systems Lab — Shenzhen, China  
**Format:** Multiple CSV files, each representing a different data dimension  
**Coverage:** 247 charging stations, 8,640 timestamps at 5-minute intervals (30 days), June 2022  

### File Structure

| File | Shape | Description |
|---|---|---|
| `time.csv` | 8,640 × 6 | Timestamp index (year, month, day, hour, minute, second) |
| `occupancy.csv` | 8,640 × 248 | Number of vehicles charging per station per timestamp |
| `volume.csv` | 8,640 × 248 | Energy volume (kWh) per station per timestamp |
| `price.csv` | 8,640 × 248 | Current price per station per timestamp |
| `duration.csv` | 8,640 × 248 | Avg session duration per station per timestamp |
| `information.csv` | 247 × 10 | Station metadata: location, charger counts, CBD flag, area |
| `adj.csv` | 247 × 248 | Station adjacency matrix (spatial graph) |
| `distance.csv` | 247 × 248 | Inter-station distances |
| `stations.csv` | 1,706 × 6 | Station coordinates |

**Integration approach:** Each matrix file (occupancy, volume, price) is melted from wide format (stations as columns) to long format (one row per station per timestamp), then merged into a single unified base table.

In [6]:
# ============================================================
# 5. URBANEV DATA — LOADING AND INTEGRATION
# ============================================================

# Load time index
time_df = pd.read_csv("../data/time.csv")
time_df["datetime"] = pd.to_datetime(
    time_df[["year", "month", "day", "hour", "minute", "second"]]
)

# Load core matrices
occupancy = pd.read_csv("../data/occupancy.csv")
volume    = pd.read_csv("../data/volume.csv")
price     = pd.read_csv("../data/price.csv")

# Load station metadata
info = pd.read_csv("../data/information.csv")
info["station_id"] = info["grid"].astype(str)

print(f"Time index: {time_df.shape} | Date range: {time_df['datetime'].min()} to {time_df['datetime'].max()}")
print(f"Occupancy matrix: {occupancy.shape}")
print(f"Volume matrix: {volume.shape}")
print(f"Price matrix: {price.shape}")
print(f"Station info: {info.shape}")

Time index: (8640, 7) | Date range: 2022-06-19 00:00:00 to 2022-07-18 23:55:00
Occupancy matrix: (8640, 248)
Volume matrix: (8640, 248)
Price matrix: (8640, 248)
Station info: (247, 11)


In [7]:
# Melt wide matrices into long format
# Each row becomes: timestamp × station_id × value

occ_long   = occupancy.melt(id_vars="timestamp", var_name="station_id", value_name="occupancy")
vol_long   = volume.melt(id_vars="timestamp", var_name="station_id", value_name="volume_kwh")
price_long = price.melt(id_vars="timestamp", var_name="station_id", value_name="price")

# Merge into unified base table
urban = occ_long.merge(vol_long,   on=["timestamp", "station_id"])
urban = urban.merge(price_long,    on=["timestamp", "station_id"])

# Add datetime
time_index = time_df[["datetime"]].reset_index().rename(columns={"index": "timestamp_idx"})
urban = urban.merge(time_index, left_on="timestamp", right_on="timestamp_idx")
urban = urban.drop(columns=["timestamp_idx"])

# Add station metadata
urban = urban.merge(
    info[["station_id", "count", "fast_count", "slow_count", "CBD", "area"]],
    on="station_id", how="left"
)

print(f"Unified UrbanEV table shape: {urban.shape}")
print(urban.head(3))
print(f"\nNull counts:\n{urban.isnull().sum()}")

Unified UrbanEV table shape: (2133833, 11)
   timestamp station_id  occupancy  volume_kwh  price            datetime  \
0          1        102         12      2.8583 0.9240 2022-06-19 00:05:00   
1          2        102         12      4.3750 0.9240 2022-06-19 00:10:00   
2          3        102         12      4.3750 0.9240 2022-06-19 00:15:00   

   count  fast_count  slow_count  CBD   area  
0     30           3          27    0 0.7100  
1     30           3          27    0 0.7100  
2     30           3          27    0 0.7100  

Null counts:
timestamp     0
station_id    0
occupancy     0
volume_kwh    0
price         0
datetime      0
count         0
fast_count    0
slow_count    0
CBD           0
area          0
dtype: int64


---
### 2.1 UrbanEV Feature Engineering

Derived features:

- **Utilization Rate** — `occupancy / count` (chargers in use / total chargers); primary demand signal
- **Demand Zone** — categorical label: `discount` (<30%), `normal` (30–80%), `surge` (>80%)
- **Occupancy Density** — `occupancy / area`; accounts for station footprint
- **Queue Length Proxy** — vehicles above 80% capacity threshold; congestion indicator
- **Time features** — hour, day of week, weekend flag for temporal modeling

In [8]:
# ============================================================
# 6. URBANEV FEATURE ENGINEERING
# ============================================================

# Utilization rate
urban["utilization_rate"] = urban["occupancy"] / urban["count"]

# Demand zone labels — maps directly to pricing agent rules
urban["demand_zone"] = pd.cut(
    urban["utilization_rate"],
    bins=[-1, 0.3, 0.8, 999],
    labels=["discount", "normal", "surge"]
)

# Time features
urban["hour"]        = urban["datetime"].dt.hour
urban["day_of_week"] = urban["datetime"].dt.dayofweek
urban["is_weekend"]  = urban["day_of_week"].isin([5, 6]).astype(int)

# Occupancy density
urban["occupancy_density"] = urban["occupancy"] / urban["area"]

# Queue length proxy — vehicles beyond 80% capacity
urban["queue_length_proxy"] = (urban["occupancy"] - urban["count"] * 0.8).clip(lower=0)

# Energy cost per kWh — use actual price data from dataset
urban["energy_cost_per_kwh"] = urban["price"]

print("UrbanEV engineered features:")
print(urban[[
    "utilization_rate", "occupancy_density",
    "queue_length_proxy", "energy_cost_per_kwh"
]].describe().round(4))

print(f"\nDemand zone distribution:\n{urban['demand_zone'].value_counts()}")

UrbanEV engineered features:
       utilization_rate  occupancy_density  queue_length_proxy  \
count      2133833.0000       2133833.0000        2133833.0000   
mean             0.2802            10.2151              0.0155   
std              0.1761            12.6291              0.2051   
min              0.0000             0.0000              0.0000   
25%              0.1515             1.9108              0.0000   
50%              0.2500             5.9211              0.0000   
75%              0.3778            14.2857              0.0000   
max              1.0769           155.2632             13.4000   

       energy_cost_per_kwh  
count         2133833.0000  
mean                0.9586  
std                 0.1813  
min                 0.2500  
25%                 0.8496  
50%                 0.9838  
75%                 1.0726  
max                 1.4700  

Demand zone distribution:
demand_zone
discount    1309011
normal       802954
surge         21868
Name: count, dty

---
## 3. Dataset Alignment Note

The two datasets serve complementary roles in this project:

| Dimension | ACN (JPL) | UrbanEV (Shenzhen) |
|---|---|---|
| Granularity | Session-level | Station × 5-minute interval |
| Geography | California, USA | Shenzhen, China |
| Primary use | Session behavior, revenue modeling | Demand forecasting, spatial analysis |
| Time coverage | 2018–2019 | June 2022 (30 days) |

They are not merged into a single table due to geographic and temporal incompatibility. Instead, they are used complementarily: UrbanEV drives the demand prediction and pricing agents; ACN provides session-level behavioral insights for EDA and business implications.

In [9]:
# ============================================================
# 7. SAVE PROCESSED OUTPUTS
# ============================================================

urban.to_csv("../data/urban_processed.csv", index=False)
acn.to_csv("../data/acn_processed.csv", index=False)

print("Outputs saved:")
print(f"  urban_processed.csv — {urban.shape[0]:,} rows × {urban.shape[1]} columns")
print(f"  acn_processed.csv   — {acn.shape[0]:,} rows × {acn.shape[1]} columns")

print("\n--- Preprocessing Complete ---")
print(f"UrbanEV date range:  {urban['datetime'].min()} to {urban['datetime'].max()}")
print(f"ACN sessions:        {len(acn):,}")
print(f"ACN avg kWh/session: {acn['kWhDelivered'].mean():.2f}")
print(f"ACN avg revenue:     ₹{acn['revenue_per_session'].mean():.2f}")

Outputs saved:
  urban_processed.csv — 2,133,833 rows × 19 columns
  acn_processed.csv   — 14,848 rows × 20 columns

--- Preprocessing Complete ---
UrbanEV date range:  2022-06-19 00:05:00 to 2022-07-18 23:55:00
ACN sessions:        14,848
ACN avg kWh/session: 8.94
ACN avg revenue:     ₹134.14
